# Sign → speech, end to end (offline, no camera needed)

**What this notebook does.** Demo + integration check, TODO §12.5/§12.6. It runs the
whole pipeline the way the deployed app will, one stage after another:

```
landmarks (543×3 per frame, NaN = undetected)
  → recognizer: the browser export (sb.recognize.export.step, .tflite), stepped frame by frame, state fed back
  → per-frame gloss + null probabilities (cosine class matrix, outside the graph)
  → OnlineDecoder: D3 segmentation + next-gloss prior (trigram) + the accept rule + noise gate
  → accepted glosses → English (sb.rescore.gloss2en rules, or a Workers AI LLM)
  → speech (Windows SAPI voice = the browser's speechSynthesis, or Workers AI TTS)
```

**Inputs.** There is no camera on this machine:
- held-out **GISLR-Sentences** streams (evaluation signers, raw landmarks with NaNs);
- optionally a **video file** (`VIDEO_FILE` in §2): MediaPipe Holistic runs in Python
  (`sb.extract.holistic.extract_video`, the same Tasks model the browser uses) and the
  landmarks go through the same chain.

**The accept rule** is read from the next-gloss notebook's selection sweep when it has
run (best selection GER for D3 + the configured prior and family). Otherwise the
config's `rule_default` is used. Either way the choice is printed.

**§3 is the integration check:** on `batch_n` held-out streams, the streaming path
(TFLite step model + `OnlineDecoder`) is compared with the offline path (PyTorch batch
forward + `fuse.decode_fused`). The two must emit the same signs, apart from float
near-ties. §3 also reports GER and per-stage latency.

**Artifacts** (`data/cache/gislr/pipeline_demo/`): `batch.json` (§3), `speech/*.wav`
(spoken outputs). The browser export is created in `registry/runs/<C1>/export/web/`
if it is missing.

## Setup

In [1]:
# ============================================================
# Imports, config, browser export, streams, prior, rule, stages
# ============================================================
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, display
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, gislr_sentences_dir
from sb.recognize import streaming as S
from sb.recognize.continuous import decode as CDEC
from sb.recognize.continuous import fuse as FU
from sb.recognize.continuous.online import OnlineDecoder
from sb.recognize.continuous.train import load_run, run_dir_for
from sb.recognize.export.step import export_web
from sb.recognize.sequences import baselines as B
from sb.recognize.sequences import metrics as M
from sb.recognize.sequences.compose import read_sequence
from sb.recognize.sequences.corpus import load_lexicon, load_sentences
from sb.rescore import client as C
from sb.rescore import gloss2en as G
from sb.rescore import prior as P
from sb.synthesize import tts

CONFIG = json.loads((EXPERIMENTS_DIR / "recognition" / "configs" / "gislr.pipeline-demo.json").read_text(encoding="utf-8"))
OUT = CACHE_DIR / "gislr" / "pipeline_demo"
(OUT / "speech").mkdir(parents=True, exist_ok=True)
torch.set_grad_enabled(False)

# ---- recognizer: the exact files the browser loads
RUN_DIR = run_dir_for(CONFIG["run"])
WEB = RUN_DIR / "export" / "web"
if not (WEB / "manifest.json").exists():
    print("browser export missing -- building it:", export_web(RUN_DIR, register=False))
MANIFEST = json.loads((WEB / "manifest.json").read_text())
import tensorflow as tf  # noqa: E402  (TFLite interpreter; the browser uses LiteRT.js on the same file)

INTERP = tf.lite.Interpreter(model_path=str(WEB / "model.tflite"))
STEP = INTERP.get_signature_runner()
W = np.fromfile(WEB / "classes.f32", np.float32).reshape(-1, MANIFEST["embed_dim"])
MASK = np.r_[np.asarray(MANIFEST["class_mask"], bool), True]
NULL, SCALE = MANIFEST["null_index"], MANIFEST["cos_scale"]
GLOSS_OF = MANIFEST["glosses"]

# ---- streams: held-out signers, identical split to every §12 notebook
ROOT = gislr_sentences_dir(CONFIG["dataset_version"])
SEQ = pd.read_csv(ROOT / "sequences.csv", keep_default_na=False)
SELECT, EVAL = B.signer_split(SEQ["participant_id"], CONFIG["n_selection_signers"], CONFIG["seed"])
EV = SEQ[(SEQ.kind == "sentence") & SEQ.participant_id.isin(EVAL)]

# ---- next-gloss prior: fitted without the stream's own sentence (its fold)
SENTS = load_sentences()
GLOSSES = sorted({g for s in SENTS for g in s.glosses})
PERM = np.array([GLOSSES.index(g) for g in GLOSS_OF])
FOLD = P.sentence_folds([s.id for s in SENTS], [s.theme for s in SENTS], CONFIG["prior"]["n_folds"], CONFIG["seed"])
_LMS = {}


def prior_for(sentence_id: str | None):
    f = FOLD.get(sentence_id, -1) if sentence_id else -1
    if f not in _LMS:
        lm = P.NgramLM(GLOSSES, CONFIG["prior"]["order"], CONFIG["prior"]["discount"]).fit(
            [s.glosses for s in SENTS if f == -1 or FOLD[s.id] != f])
        memo = {}

        def prior(hist, lm=lm, memo=memo):
            if hist not in memo:
                memo[hist] = lm.gloss_dist([GLOSS_OF[c] for c in hist])[PERM]
            return memo[hist]
        _LMS[f] = prior
    return _LMS[f]


# ---- decoder settings: D3 from the continuous eval, rule from the next-gloss sweep if it ran
d3 = next(c for c in json.loads((CACHE_DIR / "gislr/continuous_eval/results/final.json").read_text())["chosen"]
          if c["run"] == CONFIG["run"] and c["decoder"] == "D3" and c["collapse"])
NU, MIN_LEN = float(d3["nu"]), int(d3["min_len"])
sweep = CACHE_DIR / CONFIG["sweep_dir"] / f"sweep_D3_{CONFIG['rule_family']}_{CONFIG['prior']['arm']}.json"
if sweep.exists():
    best = min(json.loads(sweep.read_text()), key=lambda r: r["ger"])
    RULE = FU.Rule(**{k: best[k] for k in ("mode", "lam", "theta", "k", "theta_lo", "theta_hi", "max_len")})
    RULE_SOURCE = f"next-gloss sweep ({sweep.name}, selection GER {best['ger']:.3f})"
else:
    RULE, RULE_SOURCE = FU.Rule(**CONFIG["rule_default"]), "rule_default (next-gloss notebook not run yet)"

# ---- stage 3 + 4
LEX = load_lexicon()
LLM = None
if CONFIG["gloss2en"] != "rules_v1" and C.credentials():
    LLM = C.WorkersAI(CONFIG["llm_model"], OUT / "llm_cache.jsonl")
    PROMPT, PROMPT_SHA = C.load_prompt("v1", "gloss2en")


def to_english(glosses: list[str]) -> str:
    if LLM is not None:
        return C.clean_sentence(LLM.chat(PROMPT, PROMPT_SHA, " ".join(glosses))["text"])
    return G.convert(glosses, LEX)


def speak(text: str, name: str) -> dict:
    if CONFIG["tts"] == "sapi":
        return tts.sapi_speak(text, OUT / "speech" / f"{name}.wav", CONFIG["sapi_voice"])
    r = C.WorkersAITTS(CONFIG["workers_tts"]["model"], OUT / "speech", **CONFIG["workers_tts"]["params"]).speak(text)
    return {"path": r["path"], "seconds": r["latency_s"]}


print(f"recognizer : {WEB / 'model.tflite'} ({(WEB / 'model.tflite').stat().st_size / 1e6:.2f} MB, run {RUN_DIR.name})")
print(f"decoder    : D3 nu={NU} min_len={MIN_LEN} · prior {CONFIG['prior']['arm']} · rule {RULE.as_dict()}")
print(f"             from {RULE_SOURCE}")
print(f"English    : {'LLM ' + CONFIG['llm_model'] if LLM else 'rules_v1'} · speech: {CONFIG['tts']}")
print(f"streams    : {len(EV)} held-out sentence streams, {len(EVAL)} signers")

C:\Users\user2\repository\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


gislr_sentences_dir: kaggle copy unavailable (KaggleApiHTTPError); using the local build C:\Users\user2\repository\data\cache\gislr\sentences\v1
recognizer : C:\Users\user2\repository\registry\runs\1790143122\export\web\model.tflite (3.46 MB, run 1790143122)
decoder    : D3 nu=0.7 min_len=4 · prior trigram · rule {'mode': 'rescore', 'lam': 0.3, 'theta': 0.0, 'k': 5, 'theta_lo': 0.0, 'theta_hi': 1.01, 'max_len': None}
             from rule_default (next-gloss notebook not run yet)
English    : rules_v1 · speech: sapi
streams    : 5054 held-out sentence streams, 16 signers


## 1. Recognize one stream at a time, frame by frame

`recognize()` is the client loop: frame → TFLite step (state fed back) → class matrix
→ softmax → `OnlineDecoder.step`. A sign is **accepted at the frame its run ends**.
The printout lists the accept frame (at the dataset's ~30 fps), the gloss, and its
confidence.

In [2]:
# ============================================================
# The streaming recognizer + decoder, and a few held-out sentences through every stage
# ============================================================
N_SHOW = CONFIG["demo_n_streams"]


def frame_probs(frame: np.ndarray, state: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    o = STEP(frame=frame.astype(np.float32), state=state)
    logits = SCALE * (W @ o["embedding"])
    logits[~MASK] = -np.inf
    p = np.exp(logits - logits.max())
    return p / p.sum(), o["state_out"]


def recognize(landmarks: np.ndarray, sentence_id: str | None = None) -> dict:
    dec = OnlineDecoder(NULL, NU, MIN_LEN, RULE, prior_for(sentence_id), collapse=True)
    state = np.zeros(MANIFEST["state_shape"], np.float32)
    events, t_rec, t_dec = [], 0.0, 0.0
    for t, frame in enumerate(landmarks):
        t0 = time.perf_counter()
        p, state = frame_probs(frame, state)
        t1 = time.perf_counter()
        e = dec.step(p, t)
        t_dec += time.perf_counter() - t1
        t_rec += t1 - t0
        if e:
            events.append((*e, t))
    e = dec.flush(len(landmarks))
    if e:
        events.append((*e, len(landmarks)))
    return {"emissions": [(c, f, conf) for c, f, conf, _ in events], "events": events,
            "ms_per_frame": 1000 * t_rec / max(len(landmarks), 1), "decoder_us_per_frame": 1e6 * t_dec / max(len(landmarks), 1)}


rng = np.random.default_rng(CONFIG["seed"])
for _, row in EV.iloc[sorted(rng.choice(len(EV), N_SHOW, replace=False))].iterrows():
    lms = read_sequence(ROOT / row["npz_relpath"])["landmarks"].astype(np.float32)
    r = recognize(lms, row["sentence_id"])
    glosses = [GLOSS_OF[c] for c, _, _ in r["emissions"]]
    t0 = time.perf_counter()
    english = to_english(glosses) if glosses else ""
    t_en = time.perf_counter() - t0
    print(f"\n=== {row['seq_id']} (signer {row['participant_id']}, {len(lms)} frames) ===")
    print(f"signed    : {row['glosses']}")
    for c, f, conf, at in r["events"]:
        print(f"  accepted {GLOSS_OF[c]:<12} conf {conf:.2f} · sign ended frame {f} ({f / 30:.2f} s), decided at frame {at}")
    print(f"recognized: {' '.join(glosses) or '(nothing)'}")
    print(f"English   : {english}")
    print(f"timing    : recognizer {r['ms_per_frame']:.2f} ms/frame · decoder {r['decoder_us_per_frame']:.0f} µs/frame · English {1000 * t_en:.1f} ms")
    if english:
        s = speak(english, row["seq_id"])
        print(f"speech    : {s['seconds']:.2f} s to synthesize")
        display(Audio(s["path"]))


=== s00608 (signer 55372, 315 frames) ===
signed    : if rain yourself haveto jacket
  accepted if           conf 0.40 · sign ended frame 175 (5.83 s), decided at frame 176
  accepted rain         conf 0.76 · sign ended frame 204 (6.80 s), decided at frame 205
  accepted yourself     conf 0.95 · sign ended frame 226 (7.53 s), decided at frame 227
  accepted later        conf 0.19 · sign ended frame 237 (7.90 s), decided at frame 238
  accepted jacket       conf 0.46 · sign ended frame 303 (10.10 s), decided at frame 304
recognized: if rain yourself later jacket
English   : If it will rain, you later the jacket.
timing    : recognizer 0.07 ms/frame · decoder 3 µs/frame · English 0.1 ms


speech    : 0.23 s to synthesize



=== s04338 (signer 49445, 178 frames) ===
signed    : giraffe tongue black
  accepted giraffe      conf 0.24 · sign ended frame 70 (2.33 s), decided at frame 71
  accepted red          conf 0.60 · sign ended frame 106 (3.53 s), decided at frame 107
  accepted black        conf 0.64 · sign ended frame 168 (5.60 s), decided at frame 169
recognized: giraffe red black
English   : The giraffe is red and black.
timing    : recognizer 0.06 ms/frame · decoder 3 µs/frame · English 0.0 ms


speech    : 0.20 s to synthesize



=== s05119 (signer 30680, 77 frames) ===
signed    : lips sticky
  accepted lips         conf 0.50 · sign ended frame 37 (1.23 s), decided at frame 38
  accepted cat          conf 0.41 · sign ended frame 63 (2.10 s), decided at frame 64
recognized: lips cat
English   : The lips cat.
timing    : recognizer 0.07 ms/frame · decoder 3 µs/frame · English 0.0 ms


speech    : 0.20 s to synthesize


## 2. Optional: a video file through the same chain

Set `VIDEO_FILE` to any video of signing (e.g. recorded on a phone). MediaPipe Holistic
extracts the landmarks in Python, and the rest is §1's loop. `sentence_id` is `None`, so
the prior is the full-corpus fit.

In [3]:
# ============================================================
# Video file -> MediaPipe Holistic -> the same pipeline
# ============================================================
VIDEO_FILE = None  # e.g. Path(r"C:/Users/me/Videos/hello.mp4")

if VIDEO_FILE is None:
    print("set VIDEO_FILE to run a video through the pipeline")
else:
    from sb.extract.holistic import extract_video

    t0 = time.perf_counter()
    lms, fps = extract_video(VIDEO_FILE)
    print(f"{len(lms)} frames at {fps:.1f} fps, landmarks in {time.perf_counter() - t0:.1f} s")
    if abs(fps - 30) > 3:
        print(f"note: the model learned sign speed at GISLR's ~30 fps; this video is {fps:.0f} fps")
    r = recognize(lms)
    glosses = [GLOSS_OF[c] for c, _, _ in r["emissions"]]
    english = to_english(glosses) if glosses else ""
    print(f"recognized: {' '.join(glosses) or '(nothing)'}\nEnglish   : {english}")
    if english:
        display(Audio(speak(english, Path(VIDEO_FILE).stem)["path"]))

set VIDEO_FILE to run a video through the pipeline


## 3. Integration check on held-out streams: streaming path vs offline path

For `batch_n` seeded evaluation streams:
- **streaming**: raw landmarks → TFLite step → `OnlineDecoder` (what the client runs);
- **offline**: ME_132 features → PyTorch C1 batch forward → `fuse.segments_d3` →
  `fuse.decode_fused` → collapse (what the experiments measured).

Emissions must match. A mismatch is counted as a **near-tie** when some frame of the
stream has null probability within 1e-4 of ν, or a top-2 gloss margin under 1e-4 in a
decided segment. Only an unexplained mismatch is a bug. The notebook also reports GER
for both paths, and the English for every stream (rules, no LLM calls here).

In [4]:
# ============================================================
# Streaming (TFLite + OnlineDecoder) vs offline (PyTorch + fuse), GER, latency
# ============================================================
TIE = 1e-4

path = OUT / "batch.json"
if not path.exists():
    model, ck = load_run(RUN_DIR, torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    feats = B.StreamFeatures.build(ROOT, SEQ, np.asarray(ck["landmarks"]), ck["coords"])
    rows_ = EV.iloc[sorted(np.random.default_rng(CONFIG["seed"] + 1).choice(len(EV), CONFIG["batch_n"], replace=False))]
    sc_stream, sc_off, recs = [], [], []
    same = ties = bad = 0
    ms, us = [], []
    for i, row in tqdm(list(rows_.iterrows()), desc="streaming vs offline"):
        lms = read_sequence(ROOT / row["npz_relpath"])["landmarks"].astype(np.float32)
        r = recognize(lms, row["sentence_id"])
        ms.append(r["ms_per_frame"])
        us.append(r["decoder_us_per_frame"])
        gp, _ = CDEC.frame_outputs(model, torch.from_numpy(np.ascontiguousarray(feats.x(i))))
        off = S.collapse_repeats(FU.decode_fused(FU.segments_d3(gp, NU, MIN_LEN, NULL), prior_for(row["sentence_id"]), RULE))
        a = [(c, f) for c, f, _ in r["emissions"]]
        b = [(c, f) for c, f, _ in off]
        if a == b:
            same += 1
        else:
            srt = np.sort(gp[:, :NULL], 1)
            near = (np.abs(gp[:, NULL] - NU) < TIE).any() or ((srt[:, -1] - srt[:, -2]) < TIE).any()
            ties += near
            bad += not near
        labels, seg = B.seq_arrays(row)
        kinds = feats.kind(i)
        sc_stream.append(M.score_sequence(r["emissions"], labels, seg, kinds))
        sc_off.append(M.score_sequence(off, labels, seg, kinds))
        gl = [GLOSS_OF[c] for c, _, _ in r["emissions"]]
        recs.append({"seq_id": row["seq_id"], "signed": row["glosses"], "recognized": " ".join(gl),
                     "english": G.convert(gl, LEX) if gl else ""})
    t0 = time.perf_counter()
    for rec in recs:
        G.convert(rec["recognized"].split(), LEX)
    en_ms = 1000 * (time.perf_counter() - t0) / len(recs)
    res = {"n": len(rows_), "identical": same, "near_ties": int(ties), "unexplained": int(bad),
           "ger_streaming": M.aggregate(sc_stream)["ger"], "ger_offline": M.aggregate(sc_off)["ger"],
           "sentence_acc_streaming": M.aggregate(sc_stream)["sentence_acc"],
           "recognizer_ms_per_frame_median": float(np.median(ms)), "decoder_us_per_frame_median": float(np.median(us)),
           "english_ms_per_sentence": en_ms, "rule": RULE.as_dict(), "rule_source": RULE_SOURCE, "examples": recs}
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(res, indent=1, default=float), encoding="utf-8")
    os.replace(tmp, path)
res = json.loads(path.read_text())
print({k: v for k, v in res.items() if k != "examples"})
assert res["unexplained"] == 0, "the streaming path disagrees with the offline path beyond float near-ties"
pd.DataFrame(res["examples"]).head(15)

streaming vs offline:   0%|          | 0/300 [00:00<?, ?it/s]

{'n': 300, 'identical': 293, 'near_ties': 7, 'unexplained': 0, 'ger_streaming': 0.24505494505494504, 'ger_offline': 0.24505494505494504, 'sentence_acc_streaming': 0.47333333333333333, 'recognizer_ms_per_frame_median': 0.06171625553709534, 'decoder_us_per_frame_median': 1.8799999356576191, 'english_ms_per_sentence': 0.009741666678261632, 'rule': {'mode': 'rescore', 'lam': 0.3, 'theta': 0.0, 'k': 5, 'theta_lo': 0.0, 'theta_hi': 1.01, 'max_len': None}, 'rule_source': 'rule_default (next-gloss notebook not run yet)'}


,seq_id,signed,recognized,english
0,s00021,pretend police,pretend clean,Pretend clean.
1,s00023,uncle ride airplane tomorrow,uncle ride airplane tomorrow,The uncle will ride the airplane tomorrow.
2,s00069,goose white duck yellow,goose white duck yellow,The goose is white and yellow with the duck.
3,s00084,person man police,person man police,The person man police.
4,s00090,tomorrow snow weus stay home,tomorrow snow weus stay home,Tomorrow snow us stay home.
5,s00126,grandpa chin white hair old,grandpa chin white hair old,The Grandpa chin is white and old in hair.
6,s00136,airplane high,airplane high,The airplane is high.
7,s00219,pretend cowboy,cowboy,The cowboy.
8,s00220,if rain weus stay home,if shoe weus stay home,If our shoe stays home.
9,s00260,mom read book before sleep,farm read book,The farm reads the book.


## 4. What the client (`apps/web`) has to port

| piece | Python reference | size / cost |
|---|---|---|
| recognizer | `export/web/model.tflite` + `classes.f32` + `manifest.json` | 3.5 MB + 257 KB; 0.2 ms/frame in LiteRT.js |
| decoder | `sb.recognize.continuous.online.OnlineDecoder` | O(250) per frame |
| accept rule + gate | `sb.recognize.continuous.fuse.decide` | per accepted run |
| next-gloss prior | `sb.rescore.prior.NgramLM.to_dict()` tables | a few KB–MB JSON |
| gloss → English | `sb.rescore.gloss2en.convert` (offline) or the Worker's LLM | rules: ~ms; LLM: one call per sentence |
| speech | `speechSynthesis` (OS voices ≈ SAPI here) or Workers AI TTS | see `gislr.4.downstream.tts.ipynb` |

Each port needs a parity test against its Python reference, like §3.